In [1]:
print("Hello, World!")

Hello, World!


In [ ]:
# from ragas.metrics.collections import (
#     ContextPrecision,
#     ContextRecall,
#     ContextEntityRecall,
#     NoiseSensitivity,
#     AnswerRelevancy,
#     Faithfulness,
#     FactualCorrectness,
#     SemanticSimilarity,
# )
# from ragas.metrics.collections import (
#     MultiModalFaithfulness,
#     MultiModalRelevance,
# )
# from ragas.metrics.collections import (
#     TopicAdherence,
#     ToolCallAccuracy,
#     ToolCallF1,
#     AgentGoalAccuracyWithReference,
#     AgentGoalAccuracyWithoutReference,
# )
# from ragas.metrics.collections import (
#     FactualCorrectness,
#     SemanticSimilarity,
#     NonLLMStringSimilarity,
#     DistanceMeasure,
#     BleuScore,
#     CHRFScore,
#     RougeScore,
#     StringPresence,
#     ExactMatch,
# )

# from ragas.metrics import (
#     discrete_metric,
#     numeric_metric,
#     ranking_metric,
# )
# from ragas.metrics.collections import (
#     DomainSpecificRubrics,
#     InstanceSpecificRubrics,
# )

                             RAG
                              │
          ┌───────────────────┼───────────────────┐
          │                   │                   │
   OpenAIEmbeddings     Vector Store         Retriever
                                                  │
                                                  ▼
                                          Retrieved Docs
                                                  │
                                                  ▼
                                      ChatPromptTemplate
                                                  │
                                                  ▼
                                           ChatOpenAI
                                                  │
                                                  ▼
                                        StrOutputParser
                                                  │
                                                  ▼
                                             Response
                                                  │
                                                  │
                         RAGAS EVALUATION          │
                              │                   │
                              ▼                   ▼
                         Dataset              Response
                              │                   │
                              └─────────┬─────────┘
                                        ▼
                                   @experiment
                                        │
                         ┌──────────────┴─────────────┐
                         │                            │
                 DiscreteMetric                NumericMetric
                  Correctness                   Relevance
                         │                            │
                         └──────────────┬─────────────┘
                                        ▼
                                  Score + Reason
                                        │
                                        ▼
                                Experiment Results
                                        │
                                        ▼
                                      CSV

In [ ]:
# Rubric
# Likert

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env")

print("API key loaded successfully")

API key loaded successfully


In [3]:
MODEL = "gpt-4o-mini"

EMBEDDING_MODEL = "text-embedding-3-small"

JUDGE_MODEL = "gpt-4o-mini"

TOP_K = 3

In [4]:
from langchain_openai import (
    ChatOpenAI,
    OpenAIEmbeddings,
)

from langchain_core.documents import Document

from langchain_core.prompts import (
    ChatPromptTemplate,
)

from langchain_core.output_parsers import (
    StrOutputParser,
)

from langchain_core.vectorstores import (
    InMemoryVectorStore,
)

In [5]:
llm = ChatOpenAI(
    model=MODEL,
    temperature=0,
)

embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
)

In [6]:
llm.invoke("hi how are you?")

AIMessage(content="Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 12, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_55c03d5f8a', 'id': 'chatcmpl-EkA2o5RzELskJyyZHqt0B9XFnQC3S', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1dab1-2628-7736-a37a-d97fc7ac4256-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 29, 'total_tokens': 41, 'input_token_details': {'audio':

In [8]:
# ============================================================
# 6. KNOWLEDGE BASE
# ============================================================

DOCUMENTS = [

    Document(
        page_content="""
Ragas is an open-source evaluation framework designed
for evaluating and improving LLM applications.

It supports RAG evaluation, agent evaluation,
custom metrics, datasets, experiments,
and synthetic test data generation.
""".strip(),
        metadata={
            "doc_id": "ragas_intro",
        },
    ),

    Document(
        page_content="""
Ragas experiments allow developers to systematically
evaluate AI applications, track evaluation results,
compare different versions, and store experiment outputs.
""".strip(),
        metadata={
            "doc_id": "ragas_experiments",
        },
    ),

    Document(
        page_content="""
Ragas metrics measure different properties of AI systems.

Metrics can evaluate factual correctness,
faithfulness, answer relevancy,
retrieval quality, agent behavior,
and application-specific requirements.
""".strip(),
        metadata={
            "doc_id": "ragas_metrics",
        },
    ),

    Document(
        page_content="""
RAG stands for Retrieval-Augmented Generation.

A RAG system retrieves relevant information
from a knowledge base and provides that information
to an LLM for answer generation.
""".strip(),
        metadata={
            "doc_id": "rag_intro",
        },
    ),

]

print(
    "\nKnowledge Base Documents:",
    len(DOCUMENTS),
)


Knowledge Base Documents: 4


In [9]:
vector_store = InMemoryVectorStore(embedding=embeddings,)

vector_store.add_documents(DOCUMENTS)

['13275ebc-95e6-4afb-b7b1-2edc632f40cd',
 '42f0eb0a-0e68-4eb3-8b97-0048b276e04d',
 '3ea3f379-f089-4534-b413-df120e7e71c1',
 '21f0b781-c23c-487a-b5d5-c14d0e04fd05']

In [10]:
retriever = vector_store.as_retriever(search_kwargs={"k": TOP_K,})